In [1]:
import os
import random
import time
import pickle
import pandas as pd
import geopandas as gpd
import shapely.geometry

from tqdm import tqdm
from streetview import StreetViewDownloader, ImageService
from IPython.display import Image, display
from typing import Dict, Tuple, List, Union, Any

from streetview import *
from socioeconomics import *
from building import *
from vs30 import *
from station import *
from prompt import *
from config import *

### VS30 Data Fix for Earthquake Processing

This notebook fixes VS30 (shear-wave velocity) data in the 2019 Ridgecrest earthquake dataset by extracting accurate values from raster files.

#### Files Input
Based on the earthquake data processing pipeline, requires:

#### Earthquake Data
- `2019_ridgecrest/2019_ridgecrest_DYFI.csv` - Community-reported intensity data
- `2019_ridgecrest/stationlist.json` - Seismic station measurements

#### Geographic Files  
- `nhgis_shape/US_zcta_2019.shp` - ZIP code boundaries
- `nhgis_shape/US_blck_grp_2019_84.shp` - Census block group boundaries  
- `nhgis_shape/cbg_information.csv` - Socioeconomic attributes
- `vs30_mosaic.tif` - **VS30 raster data (key fix)**

#### Key Fix
- **Problem**: Original dataset had incorrect VS30 values
- **Solution**: Uses `rasterio` to extract VS30 values from raster at each sample location
- **Implementation**: `get_raster_value()` function performs coordinate-to-pixel lookup

1. Load 5,000 sample locations from existing prompt dataset
2. Extract accurate VS30 values using raster coordinates
3. Replace old VS30 column with raster-derived values
4. Add missing earthquake depth parameter (8.0 km)
5. Generate updated prompts with correct geotechnical data


- Fixed VS30 values for accurate site condition assessment
- Updated earthquake damage evaluation prompts
- Data leakage prevention through location identifier removal

In [2]:
import rasterio
import rasterio.windows

def get_raster_value(src, latitude, longitude):
    """
    Get the raster value at a specific latitude and longitude from an already opened raster.
    """
    # Transform latitude and longitude to pixel coordinates
    row, col = src.index(longitude, latitude)
    
    # Check if the pixel is within bounds
    if 0 <= row < src.height and 0 <= col < src.width:
        window = rasterio.windows.Window(col, row, 1, 1)
        data = src.read(1, window=window)
        return data[0][0]
    else:
        return None

In [ ]:
# Input Files
DYFI_DATA = '2014_napa/2014_napa_DYFI.csv'
STATION_DATA = '2014_napa/stationlist.json'
SOCIOECONOMIC_DATA = 'nhgis_shape/cbg_information.csv'
ZCTA_SHAPEFILE = 'nhgis_shape/US_zcta_2019.shp'
CBG_SHAPEFILE = 'nhgis_shape/US_blck_grp_2019_84.shp'
VS30_FILE = 'vs30_mosaic.tif'

# Output Files and Directories
OUTPUT_IMAGES_DIR = '2014_napa_images'
OUTPUT_IMAGES_CSV = '2014_napa_samples.csv'
CHECKPOINT_FILE = '2014_napa_checkpoint.pkl'
OUTPUT_SAMPLES_CSV = '2014_napa_samples_prompt.csv'
RAG_CHECKPOINT_FILE = '2014_napa_rag_checkpoint.pkl'
OUTPUT_RAG_SAMPLES_CSV = '2014_napa_rag_samples_prompt.csv'

# Earthquake Parameters
EARTHQUAKE_PARAMETERS = {
    "lat": 38.215,
    "lng": -122.312,
    "place": "Napa, CA",
    "magnitude": 6.0,
    "depth": 11.1,
}

# Sampling Parameters
NUM_SAMPLES = 100  # Number of sampled ZIP code
POINTS_PER_ZIP = 50  # Number of Street View samples per ZIP code
MAX_ATTEMPTS = 1000   # Maximum attempts to find valid points with Street View per ZIP code; setting a large number could result in significant cost

In [3]:
# Input Files
DYFI_DATA = '2019_ridgecrest/2019_ridgecrest_DYFI.csv'
STATION_DATA = '2019_ridgecrest/stationlist.json'
SOCIOECONOMIC_DATA = 'nhgis_shape/cbg_information.csv'
ZCTA_SHAPEFILE = 'nhgis_shape/US_zcta_2019.shp'
CBG_SHAPEFILE = 'nhgis_shape/US_blck_grp_2019_84.shp'
VS30_FILE = 'vs30_mosaic.tif'

# Output Files and Directories
OUTPUT_IMAGES_DIR = '2019_ridgecrest_images'
OUTPUT_IMAGES_CSV = '2019_ridgecrest_samples.csv'
CHECKPOINT_FILE = '2019_ridgecrest_checkpoint.pkl'
OUTPUT_SAMPLES_CSV = '2019_ridgecrest_samples_prompt.csv'
RAG_CHECKPOINT_FILE = '2019_ridgecrest_rag_checkpoint.pkl'
OUTPUT_RAG_SAMPLES_CSV = '2019_ridgecrest_rag_samples_prompt.csv'

# Earthquake Parameters
EARTHQUAKE_PARAMETERS = {
    "lat": 35.770,
    "lng": -117.599,
    "place": "Ridgecrest, CA",
    "magnitude": 7.1,
    "depth": 8.0,
}

# Sampling Parameters
NUM_SAMPLES = 100  # Number of sampled ZIP code
POINTS_PER_ZIP = 50  # Number of Street View samples per ZIP code
MAX_ATTEMPTS = 1000   # Maximum attempts to find valid points with Street View per ZIP code; setting a large number could result in significant cost

In [4]:
# INPUT ALL NECESSARY FILES

eq_data = EARTHQUAKE_PARAMETERS

zcta = gpd.read_file(ZCTA_SHAPEFILE)
zcta = zcta.to_crs(epsg=4326)

socioeconomic_df = pd.read_csv(SOCIOECONOMIC_DATA)
cbg_gdf = gpd.read_file(CBG_SHAPEFILE)
cbg_gdf = cbg_gdf.to_crs(epsg=4326)

stations_df = load_station_data(STATION_DATA)
stations_df = stations_df.sort_values(by='Nresp', ascending=False)

src = rasterio.open(VS30_FILE)

In [5]:
result_df1 = pd.read_csv('2019_ridgecrest_samples_prompt.csv')
#result_df2 = pd.read_csv('2019_ridgecrest_rag_samples_prompt.csv')

position = result_df1.columns.get_loc('eq_magnitude')
result_df1.insert(position, 'eq_depth', EARTHQUAKE_PARAMETERS['depth'])
#result_df2.insert(position, 'eq_depth', EARTHQUAKE_PARAMETERS['depth'])
result_df1.head(5)

,location_id,City,State/Region,Country,Zip Code,MMI,Responses,Latitude,Longitude,file_path,...,eq_depth,eq_magnitude,distance,vs30,population_density,urban_population_pct,median_household_income,education,over_65_rate,building
0,93555_1,Ridgecrest,CA,United States of America,93555,VII,314,35.594267,-117.694576,2019_ridgecrest_images/93555_1_google_streetvi...,...,8.0,7.1,21.36,267.58,1453.45,0.0,71953.0,34.03,18.42,A total of 37 buildings are found within a 100...
1,93555_2,Ridgecrest,CA,United States of America,93555,VII,314,35.599023,-117.693479,2019_ridgecrest_images/93555_2_google_streetvi...,...,8.0,7.1,20.84,267.58,1453.45,0.0,71953.0,34.03,18.42,A total of 32 buildings are found within a 100...
2,93555_3,Ridgecrest,CA,United States of America,93555,VII,314,35.585121,-117.664164,2019_ridgecrest_images/93555_3_google_streetvi...,...,8.0,7.1,21.38,266.16,63.19,0.0,98929.0,40.49,27.05,A total of 4 buildings are found within a 100-...
3,93555_4,Ridgecrest,CA,United States of America,93555,VII,314,35.607258,-117.761205,2019_ridgecrest_images/93555_4_google_streetvi...,...,8.0,7.1,23.28,263.93,63.19,0.0,98929.0,40.49,27.05,Building information is not available.
4,93555_5,Ridgecrest,CA,United States of America,93555,VII,314,35.585238,-117.704188,2019_ridgecrest_images/93555_5_google_streetvi...,...,8.0,7.1,22.64,270.31,63.19,0.0,98929.0,40.49,27.05,Building information is not available.


In [6]:
result_df1['vs30'] = result_df1.apply(lambda row: get_raster_value(src, row['Latitude'], row['Longitude']), axis=1)
result_df1

,location_id,City,State/Region,Country,Zip Code,MMI,Responses,Latitude,Longitude,file_path,...,eq_depth,eq_magnitude,distance,vs30,population_density,urban_population_pct,median_household_income,education,over_65_rate,building
0,93555_1,Ridgecrest,CA,United States of America,93555,VII,314,35.594267,-117.694576,2019_ridgecrest_images/93555_1_google_streetvi...,...,8.0,7.1,21.36,340,1453.45,0.0,71953.0,34.03,18.42,A total of 37 buildings are found within a 100...
1,93555_2,Ridgecrest,CA,United States of America,93555,VII,314,35.599023,-117.693479,2019_ridgecrest_images/93555_2_google_streetvi...,...,8.0,7.1,20.84,325,1453.45,0.0,71953.0,34.03,18.42,A total of 32 buildings are found within a 100...
2,93555_3,Ridgecrest,CA,United States of America,93555,VII,314,35.585121,-117.664164,2019_ridgecrest_images/93555_3_google_streetvi...,...,8.0,7.1,21.38,352,63.19,0.0,98929.0,40.49,27.05,A total of 4 buildings are found within a 100-...
3,93555_4,Ridgecrest,CA,United States of America,93555,VII,314,35.607258,-117.761205,2019_ridgecrest_images/93555_4_google_streetvi...,...,8.0,7.1,23.28,300,63.19,0.0,98929.0,40.49,27.05,Building information is not available.
4,93555_5,Ridgecrest,CA,United States of America,93555,VII,314,35.585238,-117.704188,2019_ridgecrest_images/93555_5_google_streetvi...,...,8.0,7.1,22.64,361,63.19,0.0,98929.0,40.49,27.05,Building information is not available.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,93551_48,Palmdale,CA,United States of America,93551,IV,48,34.609733,-118.261852,2019_ridgecrest_images/93551_48_google_streetv...,...,8.0,7.1,142.38,395,161.25,54.8,116742.0,34.03,18.12,A total of 9 buildings are found within a 100-...
4996,90036_49,Los Angeles,CA,United States of America,90036,IV,48,34.076025,-118.357564,2019_ridgecrest_images/90036_49_google_streetv...,...,8.0,7.1,200.65,346,2348.74,100.0,115139.0,83.33,13.92,A total of 42 buildings are found within a 100...
4997,93551_49,Palmdale,CA,United States of America,93551,IV,48,34.612779,-118.196884,2019_ridgecrest_images/93551_49_google_streetv...,...,8.0,7.1,139.68,343,5915.60,100.0,111333.0,37.69,7.94,A total of 25 buildings are found within a 100...
4998,90036_50,Los Angeles,CA,United States of America,90036,IV,48,34.079763,-118.360544,2019_ridgecrest_images/90036_50_google_streetv...,...,8.0,7.1,200.36,419,19272.34,100.0,81531.0,65.30,6.02,A total of 76 buildings are found within a 100...


In [9]:
def safe_round(value, decimals):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    try:
        return round(float(value), decimals)
    except (ValueError, TypeError):
        return "not available"

# Helper function to safely format integer values
def safe_int(value):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    try:
        return int(round(float(value), 0))
    except (ValueError, TypeError):
        return "not available"

# Helper function to safely get string values
def safe_str(value):
    if pd.isna(value) or value is None or value == "":
        return "not available"
    return str(value)


def generate_earthquake_prompts(df):
    """
    Generates system prompts and earthquake prompts using the parameter columns already added to the dataframe.
    """
    # Load checkpoint
    start_idx = 0
    result_df = df.copy()
    
    # Initialize prompt columns
    if 'system_prompt' not in result_df.columns:
        result_df['system_prompt'] = None
    
    if 'earthquake_prompt' not in result_df.columns:
        result_df['earthquake_prompt'] = None
    
    # Add system prompt
    if result_df['system_prompt'].isnull().all():
        result_df['system_prompt'] = SYSTEM_PROMPT
    
    total_rows = len(df)
    for index in range(start_idx, total_rows):
        row = result_df.iloc[index]
        
        print(f"\nGenerating prompt {index}/{total_rows}")
        
        # Generate earthquake prompt
        try:
            prompt_params = {
                "eq_place": safe_str(row['eq_place']),
                "eq_lat": safe_round(row['eq_lat'], 3),
                "eq_lng": safe_round(row['eq_lng'], 3),
                "eq_magnitude": safe_round(row['eq_magnitude'], 1),
                "eq_depth": safe_round(row['eq_depth'], 1),
                
                "state": safe_str(row['State/Region']),
                "city": safe_str(row['City']),
                "zipcode": safe_str(row['Zip Code']),
                "lat": safe_round(row['Latitude'], 3),
                "lng": safe_round(row['Longitude'], 3),
                "distance": safe_round(row['distance'], 2),
                "vs30": safe_str(row['vs30']),
                
                "population_density": safe_round(row['population_density'], 2),
                "urban_population_pct": safe_round(row['urban_population_pct'], 2),
                "median_household_income": safe_int(row['median_household_income']),
                "education": safe_round(row['education'], 2),
                "over_65_rate": safe_round(row['over_65_rate'], 2),
                
                "building": safe_str(row['building'])
            }
            
            result_df.loc[index, 'earthquake_prompt'] = EARTHQUAKE_PROMPT.format(**prompt_params)
            
        except Exception as e:
            print(f"Error generating prompt for index {index}: {str(e)}")
            # Print the problematic values to help with debugging
            print("Problematic row values:")
            for key, value in prompt_params.items():
                print(f"{key}: {value} (type: {type(value)})")
        
    return result_df

In [10]:
result_df1 = generate_earthquake_prompts(result_df1)
result_df1.to_csv(OUTPUT_SAMPLES_CSV,index=False)


Generating prompt 0/5000

Generating prompt 1/5000

Generating prompt 2/5000

Generating prompt 3/5000

Generating prompt 4/5000

Generating prompt 5/5000

Generating prompt 6/5000

Generating prompt 7/5000

Generating prompt 8/5000

Generating prompt 9/5000

Generating prompt 10/5000

Generating prompt 11/5000

Generating prompt 12/5000

Generating prompt 13/5000

Generating prompt 14/5000

Generating prompt 15/5000

Generating prompt 16/5000

Generating prompt 17/5000

Generating prompt 18/5000

Generating prompt 19/5000

Generating prompt 20/5000

Generating prompt 21/5000

Generating prompt 22/5000

Generating prompt 23/5000

Generating prompt 24/5000

Generating prompt 25/5000

Generating prompt 26/5000

Generating prompt 27/5000

Generating prompt 28/5000

Generating prompt 29/5000

Generating prompt 30/5000

Generating prompt 31/5000

Generating prompt 32/5000

Generating prompt 33/5000

Generating prompt 34/5000

Generating prompt 35/5000

Generating prompt 36/5000

Generating

In [ ]:
result_df2 = generate_earthquake_prompts(result_df2)
result_df2.to_csv(OUTPUT_RAG_SAMPLES_CSV,index=False)

### Address Data Leakage

In [18]:
df1 = pd.read_csv('2014_napa_samples_prompt.csv')
df1['earthquake_prompt'].iloc[0]

'\nThe earthquake happened date is 2025-06-01. \n\nHere is the EARTHQUAKE information. \n- Epicenter: Napa, CA\n- Coordinates: 38.215, -122.312\n- Magnitude: 6.0 mw\n- Depth: 11.1 km\n\nYOUR LOCATION information is listed below. \n- State: CA\n- City: San Francisco\n- Zipcode: 94110\n- Coordinates: 37.747, -122.411\n- Distance from epicenter: 52.78 km\n\n## Geospatial features in YOUR LOCATION\n- VS30 at your location: 450 m/s \n(VS30 represents the time-averaged shear-wave velocity (VS) to a depth of 30 meters, which is a key index to account for seismic site conditions)\n\n## Building Description in YOUR LOCATION (within a 100-meter radius)\n- Building description: A total of 108 buildings are found within a 100-meter radius, including types such as: general building (106), school (1), shed (1). Building heights range from 3.0 to 11.0 meters . \n\n## Community Socioecnomics and Demographics in YOUR LOCATION (at Cencus Block Group level)\n- Population density: 15005.73 people per squa

In [19]:
import re

def clean_prompt_text(prompt_text: str) -> str:
    """
    Removes lines related to Epicenter, State, City, and Zipcode
    from a prompt string, based on the specific formatting shown in the example.
    It aims to preserve single blank lines between blocks of information.
    """
    cleaned_text = prompt_text

    patterns_to_remove = [
        re.compile(r"^\s*-\s*Epicenter:.*$", re.IGNORECASE | re.MULTILINE),
        re.compile(r"^\s*-\s*State:.*$", re.IGNORECASE | re.MULTILINE),
        re.compile(r"^\s*-\s*City:.*$", re.IGNORECASE | re.MULTILINE),
        re.compile(r"^\s*-\s*Zipcode:.*$", re.IGNORECASE | re.MULTILINE),
    ]

    for pattern in patterns_to_remove:
        cleaned_text = pattern.sub("", cleaned_text)

    cleaned_text = re.sub(r"\n{3,}", "\n\n", cleaned_text)
    cleaned_text = cleaned_text.strip()

    return cleaned_text

In [20]:
df1['clean_prompt'] = df1['earthquake_prompt'].apply(clean_prompt_text)
df1['clean_prompt'].iloc[0]
df1.to_csv('2014_napa_samples_prompt_data_leakage_test.csv',index=False)